# 准备数据

In [7]:
# 导入模块
from datetime import datetime, date
import shelve
from functools import lru_cache

from tqdm import tqdm
import polars as pl
import rqdatac as rq

from vnpy.trader.database import DB_TZ
from vnpy.trader.datafeed import get_datafeed
from vnpy.trader.constant import Exchange, Interval
from vnpy.trader.object import HistoryRequest
from vnpy.alpha import AlphaLab, logger


In [8]:
# 配置下载参数
task_name = "csi300"
index_symbol = "000300.SSE"
rq_index_symbol = "000300.XSHG"

end_dt = datetime.now(DB_TZ)
end_date = end_dt.strftime("%Y-%m-%d")
logger.info(f"默认更新至 {end_date}")


def _normalize_listing_date(value: datetime | date | str | None) -> datetime | None:
    """将米筐返回的上市日期规范为 DB_TZ 时区"""
    if value is None:
        return None

    if isinstance(value, datetime):
        dt_obj = value
    elif isinstance(value, date):
        dt_obj = datetime.combine(value, datetime.min.time())
    elif isinstance(value, str):
        for fmt in ("%Y-%m-%d", "%Y%m%d"):
            try:
                dt_obj = datetime.strptime(value, fmt)
                break
            except ValueError:
                continue
        else:
            return None
    else:
        return None

    if dt_obj.tzinfo is None:
        return dt_obj.replace(tzinfo=DB_TZ)

    return dt_obj.astimezone(DB_TZ)


DEFAULT_INDEX_START_DT = datetime(2007, 1, 1).replace(tzinfo=DB_TZ)
index_start_dt = DEFAULT_INDEX_START_DT
start_date = index_start_dt.strftime("%Y-%m-%d")


2025-09-29 21:39:02 默认更新至 2025-09-29


In [9]:
# 创建投研实验室
lab = AlphaLab(f"./lab/{task_name}")  # 指定数据文件夹

In [15]:
# 初始化数据连接（默认使用RQData）
datafeed = get_datafeed()
datafeed.init()

KeyboardInterrupt: 

In [7]:
listing_resolved = False
try:
    index_inst = rq.instruments(rq_index_symbol)
    listing_candidate = _normalize_listing_date(
        getattr(index_inst, "listed_date", None)
    )
    if listing_candidate:
        index_start_dt = listing_candidate
        listing_resolved = True
except Exception as exc:  # noqa: BLE001
    logger.error(f"读取指数上市日期失败: {exc}")

if not listing_resolved:
    index_start_dt = DEFAULT_INDEX_START_DT
    logger.warning("指数上市日期缺失，使用 2007-01-01 作为起点")

start_date = index_start_dt.strftime("%Y-%m-%d")
logger.info(f"指数历史起始日期 {start_date}")

2025-09-29 21:24:41 指数历史起始日期 2005-04-08


In [8]:
# 增量下载指数成分股
component_fetch_start_date = datetime.strptime(start_date, "%Y-%m-%d")
component_store = lab.component_path.joinpath(index_symbol)

try:
    with shelve.open(str(component_store), flag="r") as component_db:
        if component_db:
            latest_key = max(component_db.keys())
            latest_date = datetime.strptime(latest_key, "%Y-%m-%d")
            component_fetch_start_date = max(latest_date, component_fetch_start_date)
            logger.info(
                f"指数成分已有数据截至 {latest_date:%Y-%m-%d}，将从 {component_fetch_start_date:%Y-%m-%d} 开始增量拉取"
            )
        else:
            logger.info("指数成分数据为空，将全量拉取")
except FileNotFoundError:
    logger.info("首次运行，未找到指数成分历史文件，将全量拉取")

component_fetch_start_str = component_fetch_start_date.strftime("%Y-%m-%d")
target_end_date = datetime.strptime(end_date, "%Y-%m-%d")

if component_fetch_start_date > target_end_date:
    logger.info("指数成分数据已覆盖目标结束日期，无需更新")
else:
    data = rq.index_components(
        rq_index_symbol,
        start_date=component_fetch_start_str,
        end_date=end_date,
    )

    if not data:
        logger.warning("未从米筐获得指数成分数据")
    else:
        index_components: dict[str, list[str]] = {}
        for dt, rq_symbols in data.items():
            vt_symbols: list[str] = []
            for rq_symbol in rq_symbols:
                vt_symbol = rq_symbol.replace("XSHG", "SSE").replace("XSHE", "SZSE")
                vt_symbols.append(vt_symbol)

            index_components[dt.strftime("%Y-%m-%d")] = vt_symbols

        lab.save_component_data(index_symbol, index_components)
        logger.info(f"指数成分已更新 {len(index_components)} 个交易日")


2025-09-29 21:25:03 指数成分已有数据截至 2025-09-29，将从 2025-09-29 开始增量拉取
2025-09-29 21:25:04 指数成分已更新 1 个交易日


In [10]:
# 加载指数成分股代码
component_symbols = lab.load_component_symbols(index_symbol, start_date, end_date)
print(len(component_symbols))
component_symbols[:10]

869


['600482.SSE',
 '000029.SZSE',
 '600739.SSE',
 '600748.SSE',
 '601816.SSE',
 '600456.SSE',
 '002120.SZSE',
 '603338.SSE',
 '600183.SSE',
 '600688.SSE']

In [ ]:
# 转换时间格式并确认已有数据范围
start = index_start_dt
end = end_dt


def to_rq_symbol(vt_symbol: str) -> str:
    """转换为米筐使用的合约格式"""
    return vt_symbol.replace("SSE", "XSHG").replace("SZSE", "XSHE")


@lru_cache(maxsize=None)
def get_symbol_start(vt_symbol: str) -> datetime:
    """查询单个合约的上市日期"""
    rq_symbol = to_rq_symbol(vt_symbol)
    try:
        instrument = rq.instruments(rq_symbol)
    except Exception as exc:  # noqa: BLE001
        logger.error(f"获取 {vt_symbol} 上市日期失败: {exc}")
        return start

    listing_dt = _normalize_listing_date(getattr(instrument, "listed_date", None))
    if not listing_dt:
        logger.warning(f"{vt_symbol} 缺少上市日期信息，退回指数起点 {start:%Y-%m-%d}")
        return start

    if listing_dt > end:
        logger.warning(
            f"{vt_symbol} 上市日期 {listing_dt:%Y-%m-%d} 晚于目标结束时间，跳过"
        )
        return end

    return listing_dt


def resolve_history_start(
    vt_symbol: str,
    interval: Interval,
    default_start: datetime,
) -> tuple[datetime, datetime | None]:
    """根据本地缓存决定增量下载起点"""
    if interval == Interval.DAILY:
        file_path = lab.daily_path.joinpath(f"{vt_symbol}.parquet")
    elif interval == Interval.MINUTE:
        file_path = lab.minute_path.joinpath(f"{vt_symbol}.parquet")
    else:
        return default_start, None

    if not file_path.exists():
        return default_start, None

    try:
        df = pl.read_parquet(file_path, columns=["datetime"])
    except Exception as exc:  # noqa: BLE001
        logger.error(f"读取本地 {vt_symbol} {interval.name} 数据失败: {exc}")
        return default_start, None

    if df.is_empty():
        return default_start, None

    stats = df.select(
        pl.min("datetime").alias("min_dt"),
        pl.max("datetime").alias("max_dt"),
    ).to_dict(as_series=False)

    earliest = stats["min_dt"][0]
    latest = stats["max_dt"][0]

    def ensure_db_tz(dt_obj: datetime | str | None) -> datetime | None:
        if dt_obj is None:
            return None
        if isinstance(dt_obj, str):
            dt_obj = datetime.fromisoformat(dt_obj)
        if dt_obj.tzinfo is None:
            return dt_obj.replace(tzinfo=DB_TZ)
        return dt_obj.astimezone(DB_TZ)

    earliest_dt = ensure_db_tz(earliest)
    latest_dt = ensure_db_tz(latest)

    if earliest_dt and default_start < earliest_dt:
        logger.info(
            f"{vt_symbol} 本地 {interval.name} 数据起点为 {earliest_dt:%Y-%m-%d %H:%M}，将回补更早区间"
        )
        return default_start, latest_dt

    if not latest_dt:
        return default_start, None

    request_start = max(latest_dt, default_start)
    return request_start, latest_dt


# 筛选成分股，不要把指数本身漏掉
task_symbols = component_symbols + [index_symbol]

for vt_symbol in tqdm(task_symbols, desc="增量下载K线"):
    symbol, exchange_str = vt_symbol.split(".")
    base_start = get_symbol_start(vt_symbol)

    if base_start >= end:
        logger.info(f"{vt_symbol} 上市日期晚于目标结束日期，无需下载")
        continue

    for interval in (Interval.DAILY, Interval.MINUTE):
        request_start, latest_dt = resolve_history_start(
            vt_symbol, interval, base_start
        )

        if latest_dt and latest_dt >= end:
            logger.info(
                f"{vt_symbol} {interval.name} 数据已覆盖至 {latest_dt:%Y-%m-%d %H:%M}，跳过"
            )
            continue

        req = HistoryRequest(
            symbol, Exchange(exchange_str), request_start, end, interval
        )
        bars = datafeed.query_bar_history(req)

        if bars:
            lab.save_bar_data(bars)
            logger.info(
                f"写入 {vt_symbol} {interval.name} {len(bars)} 条数据，起始 {request_start:%Y-%m-%d %H:%M}"
            )
        else:
            logger.warning(f"增量下载 {vt_symbol} {interval.name} 未返回数据")


增量下载K线:   0%|          | 0/870 [00:00<?, ?it/s]

2025-09-29 21:25:41 600482.SSE 本地 DAILY 数据起点为 2007-01-04 00:00，将回补更早区间
2025-09-29 21:25:44 写入 600482.SSE DAILY 5158 条数据，起始 2004-07-14 00:00


In [ ]:
# 添加回测参数配置
for vt_symbol in component_symbols:
    lab.add_contract_setting(
        vt_symbol,
        long_rate=5 / 10000,
        short_rate=10 / 10000,
        size=1,
        pricetick=0.0001,
    )